# Preparation

Preparation turns the raw ticket export into a trustworthy input for later enrichment and analysis. Its inputs are local API exports, and its output is a documented, prepared dataset.

---

## Libraries

Imports tools to load local JSON artifacts, inspect them as tables, and refresh the export when needed.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "preparation":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from preparation.modules import ingestion

---

## Ingestion

Downloads metadata and all ticket pages from the API, then replaces the local export files. **Run this cell only to refresh the local API export.**

In [ ]:
ingestion.ingest()

Loads the local metadata and ticket exports for exploration. They remain local because they are source artifacts.

In [ ]:
data_dir = project_root / "preparation" / "data"
metadata_path = data_dir / "metadata.json"
tickets_path = data_dir / "tickets.json"

if metadata_path.is_file() and tickets_path.is_file():
    with metadata_path.open(encoding="utf-8") as file:
        metadata = json.load(file)
    with tickets_path.open(encoding="utf-8") as file:
        tickets = json.load(file)
else:
    raise FileNotFoundError("Run the Ingestion cell before loading data.")

In [ ]:
tickets_frame = pd.DataFrame(tickets)

tickets_frame

---

## Profiling

Profiling establishes the dataset's structure, quality, coverage, and analytical limits before any cleaning rules are chosen.

### Schema

Compares the metadata's documented fields with the fields observed in the ticket export.

In [ ]:
documented_fields = metadata.get("fields", {})
observed_fields = set(tickets_frame.columns)
all_fields = sorted(set(documented_fields) | observed_fields)

schema_profile = pd.DataFrame(
    {
        "field": all_fields,
        "data_type": [
            tickets_frame[field].dtype if field in observed_fields else pd.NA
            for field in all_fields
        ],
        "documented": [field in documented_fields for field in all_fields],
        "observed": [field in observed_fields for field in all_fields],
        "description": [documented_fields.get(field) for field in all_fields],
    }
)

schema_profile

#### *Findings*

The documented `response_minutes` field is absent, while `first_response_minutes` is observed. The metadata defines the former as first-response time in minutes, and the observed field contains 8,919 integer minute values. Treat `response_minutes` → `first_response_minutes` as a provisional mapping, then retain the observed field as the canonical dataset name during cleaning.

------------------------------------------------------------------------------------------------------------------------------------

### Coverage

Counts tickets and duplicate identifiers, then parses ticket dates to establish the available time window for trend analysis.

In [ ]:
parsed_ticket_dates = pd.to_datetime(tickets_frame["ticket_date"], errors="coerce")
coverage_profile = pd.DataFrame(
    {
        "ticket_count": [len(tickets_frame)],
        "duplicate_ticket_count": [
            tickets_frame["ticket_id"].duplicated().sum()
            if "ticket_id" in tickets_frame
            else pd.NA
        ],
        "invalid_ticket_date_count": [parsed_ticket_dates.isna().sum()],
        "start_date": [parsed_ticket_dates.min()],
        "end_date": [parsed_ticket_dates.max()],
    }
)

coverage_profile

#### *Findings*

The observed unique `ticket_count` and timespan between `start_date` and `end_date` match those documented in [assignment.md](../docs/assignment.md).

---

### Data Quality

Measures missingness and data types to identify cleaning needs.

In [ ]:
quality_profile = pd.DataFrame(
    {
        "field": tickets_frame.columns,
        "data_type": tickets_frame.dtypes.astype(str).to_numpy(),
        "missing": tickets_frame.isna().sum().to_numpy(),
        "missing_rate": tickets_frame.isna().mean().to_numpy(),
    }
).sort_values(["missing_rate", "field"], ascending=[False, True])

quality_profile

#### *Findings*

All observed variables have sufficient coverage for possible analysis except `event_category` and `csat`. `response_csat` has broad coverage, but its most common value, `unresponded`, means no satisfaction response exists. Its `rated` share is approximately 15%, which aligns with the non-missing coverage of `csat`. The only observed CSAT scores are `1` and `5`, so the field behaves as a binary satisfaction outcome rather than a five-point scale. CSAT is an important outcome variable, so Analysis must decide whether to use the small rated subset, treat `response_csat` as an availability indicator, normalize the binary scores into satisfaction labels, or exclude CSAT from comparisons where sparse coverage would mislead.

---

### Categorical Distributions

Counts the values of available channels, source-system labels, and customer segments to reveal sparse or imbalanced categories.

In [ ]:
candidate_categories = [
    "channel",
    "main_contact_reason",
    "contact_reason",
    "region",
    "delivery_zone",
    "lifecycle_segment",
    "plan_size",
    "status",
    "weekday",
    "business_sla_outcome",
]
category_distributions = pd.concat(
    [
        tickets_frame[field]
        .value_counts(dropna=False)
        .rename_axis("category")
        .reset_index(name="tickets")
        .assign(field=field)
        for field in candidate_categories
        if field in tickets_frame
    ],
    ignore_index=True,
)[["field", "category", "tickets"]]

category_distributions

#### *Findings*

`channel` is not useful for comparison because 99.9% of tickets arrive by email. `delivery_zone` is imbalanced, with 89.2% local and 10.8% regional tickets, so small-group comparisons require care. `business_sla_outcome` has a meaningful minority class, with 21.8% of tickets breached, making it suitable for later exploration. `region` is too granular for direct analysis, with 181 categories and 1,019 tickets in regions represented fewer than 30 times, so it requires grouping or a minimum-volume threshold. `contact_reason` is similarly fragmented, with 80 categories and a long tail, and the metadata warns that these source-system labels may be noisy, so it should not define the final issue taxonomy. `status` is mostly `closed` at 77.3%, making it a workflow-state field rather than a strong analytical segmentation variable.

---

## Cleaning

Creates a separate prepared dataset while preserving the raw export unchanged. Cleaning makes the fields required for analysis explicit without imputing, excluding, or otherwise reinterpreting sparse values.

### Casting

Copies the raw ticket table and casts `ticket_date`, `has_churn`, `plan_size`, `first_response_minutes`, and `csat` to their analytical types. `first_response_minutes` remains the canonical observed name for the metadata's undocumented `response_minutes` field.

In [ ]:
prepared_tickets = tickets_frame.copy()
prepared_tickets["ticket_date"] = pd.to_datetime(
    prepared_tickets["ticket_date"], errors="coerce"
)
prepared_tickets["has_churn"] = prepared_tickets["has_churn"].astype("boolean")
prepared_tickets["plan_size"] = pd.to_numeric(
    prepared_tickets["plan_size"], errors="coerce"
).astype("Int64")
prepared_tickets["first_response_minutes"] = pd.to_numeric(
    prepared_tickets["first_response_minutes"], errors="coerce"
).astype("Int64")
prepared_tickets["csat"] = pd.to_numeric(
    prepared_tickets["csat"], errors="coerce"
).astype("Int64")

prepared_tickets.head()

In [ ]:
cleaning_validation = pd.DataFrame(
    {
        "check": [
            "ticket_count",
            "ticket_date_dtype",
            "has_churn_dtype",
            "plan_size_dtype",
            "first_response_minutes_dtype",
            "csat_dtype",
            "invalid_ticket_date_count",
            "source_field_count",
            "prepared_field_count",
        ],
        "value": [
            len(prepared_tickets),
            str(prepared_tickets["ticket_date"].dtype),
            str(prepared_tickets["has_churn"].dtype),
            str(prepared_tickets["plan_size"].dtype),
            str(prepared_tickets["first_response_minutes"].dtype),
            str(prepared_tickets["csat"].dtype),
            prepared_tickets["ticket_date"].isna().sum(),
            len(tickets_frame.columns),
            len(prepared_tickets.columns),
        ],
    }
)

cleaning_validation

#### *Findings*

Casting preserves all 10,000 tickets and all source fields. Every `ticket_date` parses successfully, while `has_churn` is now nullable boolean and `plan_size`, `first_response_minutes`, and `csat` are nullable integers. Sparse values remain missing rather than being imputed or excluded.

---


## Export


Writes the prepared dataset to `enrichment/data/prepared_tickets.parquet`, where the Enrichment phase begins. **Parquet preserves the cleaned dates and nullable data types** while keeping the local handoff compact. The directory remains Git-ignored.

In [ ]:
enrichment_data_dir = project_root / "enrichment" / "data"
enrichment_data_dir.mkdir(parents=True, exist_ok=True)
prepared_tickets_path = enrichment_data_dir / "prepared_tickets.parquet"
prepared_tickets.to_parquet(prepared_tickets_path, index=False)

prepared_export = pd.DataFrame(
    {
        "artifact": [prepared_tickets_path.relative_to(project_root)],
        "tickets": [len(prepared_tickets)],
        "fields": [len(prepared_tickets.columns)],
    }
)